# TPC全体のエネルギースペクトルと二次補正

ゲインマップ適用後の**イベントごとのTPC全体エネルギー**を使います。1-hitの選別はしません。
`TPC_Calibration`と同じく、同時計数で各ch組の寄与を重み付けします。未較正の組（Ka=Kb=0）は寄与0とし、その領域を含むイベントも使います。
全体のエネルギーが0のイベントは既存モジュールのSpectrumには入りません。

入力ROOTにはFADCと飛跡ヒットが必要です。ヒット未保存の古いROOT（今回のrun13など）は、この手順では使えません。

In [ ]:
from pathlib import Path
import sys,json,subprocess
import numpy as np
import matplotlib.pyplot as plt
import ROOT
ROOT.gROOT.SetBatch(True)
REPO=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'run_script/run_chain.py').exists())
sys.path.insert(0,str(REPO/'scripts/calib'))
from tpc_energy_calibration import fit_peak,gaussian_linear,quadratic_calibration,evaluate,write_correction
RAWS=[Path('/path/to/<入力raw_ROOT>')]
GAIN_MAP=Path('/path/to/<TPC_GainMapで作成したgain_map.dat>')
OUT=Path('/path/to/<出力フォルダ>')
BASE_WINDOW=30.0  # ベースラインの長さ [sample]
THRESHOLD=20.0  # 積分閾値 [ADU above baseline]
for p in [*RAWS,GAIN_MAP,OUT]:
    if '<' in str(p): raise ValueError('入力・出力パスを指定してください')
OUT.mkdir(parents=True,exist_ok=True)


## 1. ゲインマップを適用してイベントを解析
既存のTPCチェーンを使います。未較正の組がファイル中でKa=Kb=0になっていることを確認します。出力が存在すれば再利用します。ゲインや積分条件を変えるときは出力先も変えてください。

In [ ]:
gain=np.loadtxt(GAIN_MAP,ndmin=2)
if gain.shape[1]!=4 or not np.isfinite(gain).all():raise ValueError('gain mapの形式が不正です')
if np.any((gain[:,2]==0)&(gain[:,3]!=0)):raise ValueError('未較正の組はKa=Kb=0にしてください')
print('較正済みの組:',int((gain[:,2]!=0).sum()),'/',len(gain))
import hashlib
FILES=[]
for i,raw in enumerate(RAWS):
    f=ROOT.TFile.Open(str(raw))
    from sys import path
    path.insert(0,str(REPO/'scripts/analysis'))
    from tpc_raw_io import raw_tree_path
    t=f.Get(raw_tree_path(raw))
    if not t.GetBranch('hit_a_strip') or not t.GetBranch('hit_c_strip'):raise ValueError(f'{raw}: 飛跡ヒットが未保存です')
    f.Close()
    target=OUT/f'energy_{i:03d}.root'
    cmd=[sys.executable,str(REPO/'run_script/run_chain.py'),'smile3_tpc_energy','--root-input',str(raw),'-o',str(target),
         '--set','TPC_Calibration.NoCalData=false','--set',f'TPC_Calibration.ParameterFile={GAIN_MAP}',
         '--set','TPC_Calibration.Spectrum_TH=true','--set',f'CalcFADC.base_window={BASE_WINDOW}',
         '--set',f'CalcFADC.threshold={THRESHOLD}']
    print(' '.join(cmd))
    provenance={'command':cmd,'gain_map_sha256':hashlib.sha256(GAIN_MAP.read_bytes()).hexdigest(),'raw_size':raw.stat().st_size,'raw_mtime_ns':raw.stat().st_mtime_ns}
    stamp=target.with_suffix('.inputs.json')
    if target.exists() and (not stamp.exists() or json.loads(stamp.read_text())!=provenance):
        raise RuntimeError(f'{target}: 入力または条件を確認できません。別の出力フォルダを指定してください')
    if not target.exists():
        with target.with_suffix('.log').open('w') as log:subprocess.run(cmd,stdout=log,stderr=subprocess.STDOUT,check=True)
        stamp.write_text(json.dumps(provenance,ensure_ascii=False,indent=2))
    FILES.append(target)


## 2. 全イベントのエネルギースペクトル
各runのイベントスペクトルを合計します。未較正領域の寄与0により、部分的なエネルギーになるイベントも含まれます。

In [ ]:
hist=None
for target in FILES:
    f=ROOT.TFile.Open(str(target));h=f.Get('TPC_Calibration/Spectrum')
    if not h:raise RuntimeError(f'{target}: Spectrumがありません')
    print(target,'entries=',h.GetEntries(),'underflow=',h.GetBinContent(0),'overflow=',h.GetBinContent(h.GetNbinsX()+1))
    if hist is None:hist=h.Clone('Spectrum_gain');hist.SetDirectory(0)
    else:hist.Add(h)
    f.Close()
edges=np.array([hist.GetXaxis().GetBinLowEdge(i) for i in range(1,hist.GetNbinsX()+2)])
counts=np.array([hist.GetBinContent(i) for i in range(1,hist.GetNbinsX()+1)])
x=(edges[:-1]+edges[1:])/2
fig,axs=plt.subplots(2,1,figsize=(7,10),sharex=True)
for ax in axs:
    ax.step(x,counts,where='mid');ax.set(xlim=(0,120),ylabel='events / bin')
    for e in (8,30,80):ax.axvline(e,ls=':',color='gray')
axs[0].set_title('TPC energy after gain map / all events')
axs[1].set_yscale('log');axs[1].set_ylim(bottom=.8);axs[1].set_xlabel('TPC energy [keV]')
fig.tight_layout();fig.savefig(OUT/'spectrum_gain.png',dpi=150);plt.show()
f=ROOT.TFile(str(OUT/'spectrum_gain.root'),'RECREATE');hist.Write();f.Close()


## 3. Cu・Baの3ピークをフィット

ガウス＋直線背景で中心と標準誤差を求めます。範囲は図を見て調整してください。
基準値は候補です。Cu Kαと約30 keVのX線群は重なった線の有効エネルギーをどう扱うか確認し、`REFERENCES_CONFIRMED=True`にしてから保存します。
Cuの線は [NIST](https://physics.nist.gov/cgi-bin/XrayTrans/search.pl?element=Cu&trans=All&units=eV)、Ba-133の核データは [LNHB](https://www.lnhb.fr/nuclides/Ba-133_com.pdf) を参照。


In [ ]:
REFERENCE_KEV=np.array([8.04,30.85,80.9979])  # 採用する基準エネルギー [keV]。要確認
REFERENCES_CONFIRMED=False
WINDOWS=[(4.,12.),(20.,42.),(55.,105.)]  # ゲイン適用後のfit範囲 [keV]
CHI2NDF_MAX=5.0  # χ²/ndfの上限 [dimensionless]
SIGNIFICANCE_MIN=3.0  # ガウス高さ/標準誤差の下限 [dimensionless]
fits=[fit_peak(edges,counts,w,CHI2NDF_MAX,SIGNIFICANCE_MIN) for w in WINDOWS]
fig,axs=plt.subplots(3,1,figsize=(7,12))
for ax,r,eref in zip(axs,fits,REFERENCE_KEV):
    sel=(x>=r['window'][0])&(x<=r['window'][1]);ax.errorbar(x[sel],counts[sel],np.sqrt(np.maximum(counts[sel],1)),fmt='.',ms=3)
    xx=np.linspace(*r['window'],300);ax.plot(xx,gaussian_linear(xx,*r['parameters']))
    ax.set(title=f"reference {eref:g} keV / mean {r['mean']:.3f} ± {r['mean_error']:.3f}",xlabel='measured energy [keV]',ylabel='events / bin')
    print(r)
fig.tight_layout();fig.savefig(OUT/'peak_fits.png',dpi=150);plt.show()


## 4. 全体の二次補正

`E_corrected = a0 + a1 E_gain + a2 E_gain²`。3点を通すため自由度は0で、補正式のχ²による検証はできません。
ピーク中心の標準誤差を係数の3×3共分散に一次伝播します（ピーク間独立・基準エネルギー固定）。系統誤差は含みません。
3ピークの間で単調増加することを確認します。範囲外は端点の接線で連続的に延長し、負のエネルギーは0とします。

In [ ]:
if not all(r['accepted'] for r in fits):raise RuntimeError('採用条件に通らないピークがあります。fit範囲・ピーク形状を確認してください')
cal=quadratic_calibration([r['mean'] for r in fits],[r['mean_error'] for r in fits],REFERENCE_KEV)
cal.update(gain_map=str(GAIN_MAP.resolve()),inputs=[str(p.resolve()) for p in RAWS],base_window_samples=BASE_WINDOW,threshold_ADU=THRESHOLD,selection='all events; uncalibrated contributions zero',peak_fits=fits)
print(json.dumps(cal,ensure_ascii=False,indent=2))
fig,ax=plt.subplots(figsize=(6,5));ax.errorbar(cal['means_keV'],REFERENCE_KEV,xerr=cal['mean_errors_keV'],fmt='o')
xx=np.linspace(0,120,500);ax.plot(xx,evaluate(xx,cal));ax.plot(xx,xx,ls=':',color='gray');ax.set(xlabel='gain-map energy [keV]',ylabel='corrected energy [keV]');fig.tight_layout();fig.savefig(OUT/'energy_correction.png',dpi=150);plt.show()


## 5. 補正後のスペクトルと保存
ここでの補正後ヒストグラムは入力エネルギービンの中心を変換した表示です。イベントごとの補正結果と比べ、入力ビン幅（0.2 keV）に由来する丸めがあります。較正ファイルは基準値の確認後だけ出力します。

In [ ]:
corrected=np.histogram(evaluate(x,cal),bins=edges,weights=counts)[0]
fig,ax=plt.subplots(figsize=(7,9));ax.step(x,counts,where='mid',label='gain map');ax.step(x,corrected,where='mid',label='quadratic corrected')
for e in REFERENCE_KEV:ax.axvline(e,ls=':',color='gray')
ax.set(xlim=(0,120),xlabel='energy [keV]',ylabel='events / bin',yscale='log');ax.legend();fig.tight_layout();fig.savefig(OUT/'spectrum_before_after.png',dpi=150);plt.show()
if not REFERENCES_CONFIRMED:raise RuntimeError('REFERENCE_KEVを確認し、REFERENCES_CONFIRMED=Trueにしてから保存してください')
write_correction(OUT/'tpc_energy_correction.dat',cal)
print('wrote',OUT/'tpc_energy_correction.dat')


## 6. 再構成チェーンへの適用

ゲインマップと全体補正を両方指定します。補正は各イベントのTPC全体エネルギーに一度だけ適用されます。

```bash
python3 run_script/run_chain.py smile3_tpc_energy --root-input "<入力raw_ROOT>" -o "<出力ROOT>" \
    --set TPC_Calibration.NoCalData=false \
    --set TPC_Calibration.ParameterFile="<gain_map.dat>" \
    --set TPC_Calibration.energy_correction_file="<tpc_energy_correction.dat>"
```

`smile3_reconstruct`でも同じ指定で適用できます。Spectrumが補正後、Spectrum_GainMapが補正前です。
補正ファイルの係数誤差はJSONに保存します。チェーンのイベントエネルギーへの誤差伝播は行いません。
